<a href="https://colab.research.google.com/github/AllyApitchaya/msc-adr-prediction/blob/main/notebooks/08_sider_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/ADR_Project'
print(f'Project: {PROJECT}')

Mounted at /content/drive
Project: /content/drive/MyDrive/ADR_Project


In [2]:
# load our data so we can get drug names and PubChem CIDs
train_df = pd.read_csv(os.path.join(PROJECT, 'data/processed/train.csv'))
test_df  = pd.read_csv(os.path.join(PROJECT, 'data/processed/test.csv'))

all_df = pd.concat([train_df, test_df], ignore_index=True)
drugs = all_df['drug'].unique()
adrs  = all_df['adr'].unique()
print(f'{len(drugs)} drugs, {len(adrs)} unique ADR terms')

# load model predictions from the test set
test_metrics = pd.read_csv(os.path.join(PROJECT, 'results/multimodal_test_metrics.csv'))
print(f'\nModel test metrics:')
print(test_metrics.to_string(index=False))

15 drugs, 3561 unique ADR terms

Model test metrics:
   AUROC    AUPRC  Sensitivity  Precision  Accuracy
0.850903 0.857594     0.734694   0.810573    0.7815


In [3]:
# download SIDER side-effect data
# this file has all drug-side effect pairs from drug labels
sider_url = 'http://sideeffects.embl.de/media/download/meddra_all_se.tsv.gz'

sider = pd.read_csv(
    sider_url, sep='\t', compression='gzip', header=None,
    names=['stitch_id', 'umls_cui_label', 'meddra_type', 'umls_cui_meddra', 'side_effect']
)

print(f'SIDER raw: {len(sider):,} drug-side effect pairs')
print(f'Unique drugs in SIDER: {sider["stitch_id"].nunique():,}')
print(f'Unique side effects: {sider["side_effect"].nunique():,}')
print(f'MedDRA types: {sider["meddra_type"].unique().tolist()}')
sider.head()

SIDER raw: 309,849 drug-side effect pairs
Unique drugs in SIDER: 1,556
Unique side effects: 6,123
MedDRA types: ['LLT', 'PT', nan]


,stitch_id,umls_cui_label,meddra_type,umls_cui_meddra,side_effect
CID100000085,CID000010917,C0000729,LLT,C0000729,Abdominal cramps
CID100000085,CID000010917,C0000729,PT,C0000737,Abdominal pain
CID100000085,CID000010917,C0000737,LLT,C0000737,Abdominal pain
CID100000085,CID000010917,C0000737,PT,C0687713,Gastrointestinal pain
CID100000085,CID000010917,C0000737,PT,C0000737,Abdominal pain


In [4]:
# map our 15 drugs to STITCH compound IDs
# STITCH uses PubChem CIDs with a prefix: CID1 (stereo) or CID0 (flat)
drug_cids = all_df[['drug', 'pubchem_cid']].drop_duplicates().reset_index(drop=True)
drug_cids['cid_int'] = drug_cids['pubchem_cid'].astype(int)
print('Drug to PubChem CID:')
for _, row in drug_cids.iterrows():
    print(f'  {row["drug"]:20s} CID={int(row["cid_int"]):>10d}')

Drug to PubChem CID:
  glipizide            CID=      3478
  nateglinide          CID=   5311309
  metformin            CID=      4091
  glyburide            CID=      3488
  dapagliflozin        CID=   9887712
  saxagliptin          CID=  11243969
  acarbose             CID=   9811704
  repaglinide          CID=     65981
  rosiglitazone        CID=     77999
  sitagliptin          CID=   4369359
  pioglitazone         CID=      4829
  canagliflozin        CID=  24812758
  linagliptin          CID=  10096344
  empagliflozin        CID=  11949646
  glimepiride          CID=      3476


In [5]:
# check which of our drugs are actually in SIDER
# try both flat and stereo IDs since SIDER might use either
sider['cid_int'] = sider['stitch_id'].str[4:].astype(int)
sider_cids = set(sider['cid_int'].unique())
drug_stitch_map = {}
for _, row in drug_cids.iterrows():
    cid = int(row['cid_int'])
    if cid in sider_cids:
        drug_stitch_map[row['drug']] = cid
found = set(drug_stitch_map.keys()); missing = set(drugs) - found
print(f'Drugs found in SIDER: {len(found)}/{len(drugs)}')
for d, cid in sorted(drug_stitch_map.items()):
    n_se = sider[sider['cid_int'] == cid]['side_effect'].nunique()
    print(f'  {d:20s} -> CID{cid}  ({n_se} side effects)')
if missing:
    print(f'\nNot found in SIDER: {sorted(missing)}')

Drugs found in SIDER: 13/15
  canagliflozin        -> CID24812758  (93 side effects)
  dapagliflozin        -> CID9887712  (83 side effects)
  empagliflozin        -> CID11949646  (56 side effects)
  glimepiride          -> CID3476  (126 side effects)
  glipizide            -> CID3478  (106 side effects)
  glyburide            -> CID3488  (102 side effects)
  linagliptin          -> CID10096344  (39 side effects)
  metformin            -> CID4091  (141 side effects)
  nateglinide          -> CID5311309  (46 side effects)
  pioglitazone         -> CID4829  (121 side effects)
  repaglinide          -> CID65981  (66 side effects)
  saxagliptin          -> CID11243969  (58 side effects)
  sitagliptin          -> CID4369359  (129 side effects)

Not found in SIDER: ['acarbose', 'rosiglitazone']


In [6]:
# build the SIDER reference set: known (drug, side_effect) pairs
# normalize side effect names to lowercase for matching
sider_pairs = set()
for drug, cid in drug_stitch_map.items():
    for se in sider[sider['cid_int'] == cid]['side_effect'].str.lower().unique():
        sider_pairs.add((drug, se))
print(f'Total SIDER (drug, side_effect) pairs for our drugs: {len(sider_pairs):,}')

Total SIDER (drug, side_effect) pairs for our drugs: 1,166


In [7]:
# get our model's positive predictions from the test set
# the test_df has labels; we need to identify which pairs the model predicts as positive
# since we saved metrics but not raw predictions, reconstruct from test_df
# positive labels in test_df = drug-ADR associations the model should predict

# for external validation we use ALL known positives from our dataset (train + test)
# these are the associations our model is designed to detect
our_positives = all_df[all_df['label'] == 1][['drug', 'adr']].drop_duplicates()
our_positives['adr_lower'] = our_positives['adr'].str.lower()

print(f'Our dataset positive (drug, ADR) pairs: {len(our_positives):,}')
print(f'Unique drugs in positives: {our_positives["drug"].nunique()}')
print(f'Unique ADRs in positives: {our_positives["adr"].nunique()}')

Our dataset positive (drug, ADR) pairs: 6,621
Unique drugs in positives: 15
Unique ADRs in positives: 3350


In [8]:
# cross-reference: check vocabulary overlap first
# our ADR terms might not match SIDER's MedDRA terms exactly
our_adr_terms = set(our_positives['adr_lower'].unique())
sider_se_terms = set(se for _, se in sider_pairs)

vocab_overlap = our_adr_terms & sider_se_terms
our_only = our_adr_terms - sider_se_terms
sider_only = sider_se_terms - our_adr_terms

print(f'Vocabulary comparison:')
print(f'  Our ADR terms:     {len(our_adr_terms):,}')
print(f'  SIDER SE terms:    {len(sider_se_terms):,}')
print(f'  Overlapping terms: {len(vocab_overlap):,} ({len(vocab_overlap)/len(our_adr_terms):.1%} of ours)')
print(f'  Our terms not in SIDER: {len(our_only):,}')

if len(vocab_overlap) < 20:
    print(f'\nOverlapping terms: {sorted(vocab_overlap)}')

Vocabulary comparison:
  Our ADR terms:     3,350
  SIDER SE terms:    458
  Overlapping terms: 328 (9.8% of ours)
  Our terms not in SIDER: 3,022


In [9]:
# also try fuzzy matching for terms that differ slightly
# common differences: plural/singular, hyphenation, word order

from difflib import get_close_matches

sider_se_list = sorted(sider_se_terms)
fuzzy_map = {}  # our_term -> sider_term

for term in sorted(our_only):
    matches = get_close_matches(term, sider_se_list, n=1, cutoff=0.85)
    if matches:
        fuzzy_map[term] = matches[0]

print(f'Fuzzy matches found: {len(fuzzy_map)}')
for ours, theirs in sorted(fuzzy_map.items())[:20]:
    print(f'  "{ours}" -> "{theirs}"')

if len(fuzzy_map) > 20:
    print(f'  ... and {len(fuzzy_map) - 20} more')

Fuzzy matches found: 99
  "abdominal pain lower" -> "abdominal pain upper"
  "acute myocardial infarction" -> "myocardial infarction"
  "adrenal disorder" -> "retinal disorder"
  "alcohol use" -> "alcohol abuse"
  "anal haemorrhage" -> "retinal haemorrhage"
  "anaphylactic reaction" -> "anaphylactoid reaction"
  "angioplasty" -> "angiopathy"
  "aspartate aminotransferase increased" -> "alanine aminotransferase increased"
  "asthenopia" -> "asthenia"
  "autoimmune nephritis" -> "autoimmune hepatitis"
  "bandaemia" -> "anaemia"
  "biliary tract disorder" -> "urinary tract disorder"
  "blood creatine increased" -> "blood creatinine increased"
  "blood creatinine decreased" -> "blood creatinine increased"
  "blood cyanide increased" -> "blood creatinine increased"
  "blood glucagon increased" -> "blood glucose increased"
  "blood iron increased" -> "blood urea increased"
  "blood lead increased" -> "blood urea increased"
  "blood phosphorus decreased" -> "blood phosphorus increased"
  "blo

In [10]:
# build the final mapping: exact + fuzzy
def normalize_adr(term, fuzzy_map):
    """Map our ADR term to SIDER vocabulary if possible."""
    t = term.lower()
    if t in sider_se_terms:
        return t
    if t in fuzzy_map:
        return fuzzy_map[t]
    return None

# cross-reference our positive pairs against SIDER
# only consider drugs that are in SIDER
results = []

for _, row in our_positives.iterrows():
    drug = row['drug']
    adr = row['adr']
    adr_lower = row['adr_lower']

    if drug not in drug_stitch_map:
        results.append({'drug': drug, 'adr': adr, 'in_sider': 'drug_not_in_sider',
                        'matched_term': None})
        continue

    mapped = normalize_adr(adr, fuzzy_map)
    if mapped is None:
        results.append({'drug': drug, 'adr': adr, 'in_sider': 'term_not_mapped',
                        'matched_term': None})
        continue

    if (drug, mapped) in sider_pairs:
        results.append({'drug': drug, 'adr': adr, 'in_sider': 'confirmed',
                        'matched_term': mapped})
    else:
        results.append({'drug': drug, 'adr': adr, 'in_sider': 'not_in_sider',
                        'matched_term': mapped})

results_df = pd.DataFrame(results)
print(f'Cross-reference results:')
print(results_df['in_sider'].value_counts().to_string())

Cross-reference results:
in_sider
term_not_mapped      4887
not_in_sider         1147
confirmed             539
drug_not_in_sider      48


In [11]:
# compute validation metrics
# only meaningful for the subset where both drug and ADR term are mappable

mappable = results_df[results_df['in_sider'].isin(['confirmed', 'not_in_sider'])]
confirmed = results_df[results_df['in_sider'] == 'confirmed']

total_positive_pairs = len(our_positives)
n_drug_in_sider = len(set(drug_stitch_map.keys()))
n_mappable = len(mappable)
n_confirmed = len(confirmed)

print(f'Validation summary:')
print(f'  Total positive pairs in our data: {total_positive_pairs}')
print(f'  Drugs found in SIDER: {n_drug_in_sider}/{len(drugs)}')
print(f'  Pairs where both drug and ADR are mappable: {n_mappable}')
print(f'  Pairs confirmed by SIDER: {n_confirmed}')
print()

# precision vs SIDER: of our positive predictions that are mappable, how many does SIDER confirm?
precision_vs_sider = n_confirmed / n_mappable if n_mappable > 0 else 0
print(f'Precision vs SIDER (confirmed / mappable positives): {precision_vs_sider:.4f}')

# recall on SIDER: of SIDER's known pairs for our drugs, how many do we capture?
# restrict to SIDER pairs where the side effect is in our vocabulary
sider_in_our_vocab = set()
reverse_fuzzy = {v: k for k, v in fuzzy_map.items()}

for drug, se in sider_pairs:
    if drug not in drug_stitch_map:
        continue
    # check if this SIDER term maps to any of our terms
    if se in our_adr_terms or se in reverse_fuzzy:
        sider_in_our_vocab.add((drug, se))

recall_on_sider = n_confirmed / len(sider_in_our_vocab) if len(sider_in_our_vocab) > 0 else 0
print(f'Recall on SIDER (confirmed / SIDER pairs in our vocab): {recall_on_sider:.4f}')
print(f'  ({n_confirmed} / {len(sider_in_our_vocab)} SIDER pairs with matching terms)')

# coverage: what fraction of our pairs can even be checked?
coverage = n_mappable / total_positive_pairs if total_positive_pairs > 0 else 0
print(f'\nCoverage (mappable / total): {coverage:.4f}')
print(f'  Note: limited by vocabulary differences between our ADR terms and SIDER/MedDRA')

Validation summary:
  Total positive pairs in our data: 6621
  Drugs found in SIDER: 13/15
  Pairs where both drug and ADR are mappable: 1686
  Pairs confirmed by SIDER: 539

Precision vs SIDER (confirmed / mappable positives): 0.3197
Recall on SIDER (confirmed / SIDER pairs in our vocab): 0.5698
  (539 / 946 SIDER pairs with matching terms)

Coverage (mappable / total): 0.2546
  Note: limited by vocabulary differences between our ADR terms and SIDER/MedDRA


In [12]:
# per-drug breakdown
print(f'{"Drug":20s} {"Positives":>10s} {"Mappable":>10s} {"Confirmed":>10s} {"Prec":>8s}')
print('-' * 62)

per_drug = []
for drug in sorted(drugs):
    drug_rows = results_df[results_df['drug'] == drug]
    n_pos = len(drug_rows)
    n_map = len(drug_rows[drug_rows['in_sider'].isin(['confirmed', 'not_in_sider'])])
    n_conf = len(drug_rows[drug_rows['in_sider'] == 'confirmed'])
    prec = n_conf / n_map if n_map > 0 else float('nan')
    per_drug.append({'drug': drug, 'positives': n_pos, 'mappable': n_map,
                     'confirmed': n_conf, 'precision_vs_sider': prec})
    print(f'{drug:20s} {n_pos:>10d} {n_map:>10d} {n_conf:>10d} {prec:>8.3f}' if n_map > 0
          else f'{drug:20s} {n_pos:>10d} {n_map:>10d} {n_conf:>10d} {"N/A":>8s}')

per_drug_df = pd.DataFrame(per_drug)

Drug                  Positives   Mappable  Confirmed     Prec
--------------------------------------------------------------
acarbose                     41          0          0      N/A
canagliflozin               372        120         32    0.267
dapagliflozin               994        264         73    0.277
empagliflozin               614        201         39    0.194
glimepiride                 296        110         52    0.473
glipizide                   270        102         45    0.441
glyburide                    70         31         13    0.419
linagliptin                 349        125         16    0.128
metformin                  2853        402        131    0.326
nateglinide                   8          7          5    0.714
pioglitazone                284        114         55    0.482
repaglinide                 121         56         19    0.339
rosiglitazone                 7          0          0      N/A
saxagliptin                  69         45         12  

In [13]:
# save everything
results_dir = os.path.join(PROJECT, 'results')
os.makedirs(results_dir, exist_ok=True)

# summary metrics
summary = pd.DataFrame([{
    'n_drugs_total': len(drugs),
    'n_drugs_in_sider': n_drug_in_sider,
    'n_positive_pairs': total_positive_pairs,
    'n_mappable_pairs': n_mappable,
    'n_confirmed_by_sider': n_confirmed,
    'precision_vs_sider': precision_vs_sider,
    'recall_on_sider': recall_on_sider,
    'coverage': coverage,
    'n_vocab_overlap': len(vocab_overlap),
    'n_fuzzy_matches': len(fuzzy_map),
}])

summary.to_csv(os.path.join(results_dir, 'sider_validation.csv'), index=False)
per_drug_df.to_csv(os.path.join(results_dir, 'sider_validation_per_drug.csv'), index=False)
results_df.to_csv(os.path.join(results_dir, 'sider_validation_pairs.csv'), index=False)

print('Saved to results/:')
print('  sider_validation.csv          -- summary metrics')
print('  sider_validation_per_drug.csv -- per-drug breakdown')
print('  sider_validation_pairs.csv    -- all pair-level results')
print()
print('Summary:')
print(summary.T.to_string(header=False))

Saved to results/:
  sider_validation.csv          -- summary metrics
  sider_validation_per_drug.csv -- per-drug breakdown
  sider_validation_pairs.csv    -- all pair-level results

Summary:
n_drugs_total           15.000000
n_drugs_in_sider        13.000000
n_positive_pairs      6621.000000
n_mappable_pairs      1686.000000
n_confirmed_by_sider   539.000000
precision_vs_sider       0.319692
recall_on_sider          0.569767
coverage                 0.254644
n_vocab_overlap        328.000000
n_fuzzy_matches         99.000000
